In [9]:
# imports
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report, confusion_matrix)


In [10]:
df = pd.read_csv("/home/desktop/Optimizing-P2P-Loan-Thresholds-for-Borrowers-Using-LendingClub-Data-/dataset/clean_undersampled.csv", low_memory=False)

In [11]:
# features / target
y = df['profitable']
X = df.drop(columns=['profitable'])
print(X.shape)
print("class balance:\n", y.value_counts())

(549174, 151)
class balance:
 profitable
0    274587
1    274587
Name: count, dtype: int64


In [12]:
# same split as XGBoost notebook
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)

# validation split off the training portion
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1765, random_state=42, stratify=y_train)

print(len(X_train), len(X_val), len(X_test))
print("class balance per split:", y_train.mean(), y_val.mean(), y_test.mean())  # all ~0.5

384407 82390 82377
class balance per split: 0.5000013007047218 0.5 0.499993930344635


In [ ]:
# scale each feature so that large values, like loan amount dont expload in distance
# distance = sqrt(Δgrade² + Δloan_amnt² + Δannual_inc² + ...)
# SVM in one object so the scaler only fits on training data

# SVM training do not take NaN vlaues. SimpleImputer i used to fill those columns

svm_pipeline = make_pipeline(
    
    SimpleImputer(strategy='median'),  # robust to the skewed money columns
    StandardScaler(),      # mean=0, std=1 for every feature
    SVC(kernel='linear')  # linear SVM, default C=1
)

# fit ONLY on the training portion
svm_pipeline.fit(X_train, y_train)
print("fitted. support vectors:", svm_pipeline.named_steps['svm'].n_support_)

In [ ]:
y_pred = svm_pipeline.predict(X_val)

print("Accuracy:", accuracy_score(y_val, y_pred))
print("Precision:", precision_score(y_val, y_pred))
print("Recall:   ", recall_score(y_val, y_pred))
print("F1:       ", f1_score(y_val, y_pred))
print("ROC AUC:  ", roc_auc_score(y_val, svm_pipeline.predict_proba(X_val)[:, 1]))
print()
print(classification_report(y_val, y_pred))
print(confusion_matrix(y_val, y_pred))